# Part 4: Add Work IQ

In Part 3, you added structured data from Fabric IQ. Now you'll add the first-party `workIQ` knowledge source so the knowledge base can retrieve workplace context alongside the restored HR and health search indexes.

Work IQ retrieval can take longer than index retrieval, so this notebook gives the retrieve request extra runtime.

> **Before you start:** Run `az login --tenant <your-tenant-id>` in a terminal before starting this notebook. This notebook uses `AzureCliCredential` to request delegated query-source tokens from your Azure CLI sign-in.

## Step 1: Set up environment variables

Load the configuration for your Azure AI Search and Azure OpenAI resources. The `.env` file in the project folder has the values needed for this part.

In [ ]:
import json
import os

from azure.core.credentials import AzureKeyCredential
from azure.identity import AzureCliCredential
from dotenv import load_dotenv

load_dotenv(override=True)


AZURE_SEARCH_SERVICE_ENDPOINT = os.environ["AZURE_SEARCH_SERVICE_ENDPOINT"]
AZURE_SEARCH_ADMIN_KEY = os.environ["AZURE_SEARCH_ADMIN_KEY"]
AZURE_OPENAI_ENDPOINT = os.environ["AZURE_OPENAI_ENDPOINT"]
AZURE_SEARCH_API_VERSION = "2026-05-01-preview"
AZURE_OPENAI_KEY = os.environ["AZURE_OPENAI_KEY"]
AZURE_OPENAI_CHATGPT_DEPLOYMENT = os.environ["AZURE_OPENAI_CHATGPT_DEPLOYMENT"]
AZURE_OPENAI_CHATGPT_MODEL_NAME = os.environ["AZURE_OPENAI_CHATGPT_MODEL_NAME"]
AZURE_TENANT_ID = os.environ["AZURE_TENANT_ID"]

HRDOCS_INDEX = "hrdocs"
HEALTHDOCS_INDEX = "healthdocs"

search_credential = AzureKeyCredential(AZURE_SEARCH_ADMIN_KEY)
query_source_credential = AzureCliCredential(tenant_id=AZURE_TENANT_ID, process_timeout=60)

print("Environment variables loaded")


## Step 2: Get the Search query-source token

The Work IQ source is queried at runtime using a delegated token. In this step, get a Search query-source token for the target tenant so the knowledge base can access Work IQ during retrieval.

This notebook uses `AzureCliCredential`, so the token comes from the Azure CLI account you signed in with before starting the notebook.

In [ ]:
QUERY_SOURCE_AUTH = query_source_credential.get_token("https://search.azure.com/.default").token
print("Acquired token for Azure AI Search query source")


## Step 3: Create three knowledge sources

For this knowledge base, you'll create three knowledge sources: the same two search-index sources used earlier, plus a Work IQ knowledge source.

* `hrdocs-knowledge-source`: Points to the `hrdocs` search index
* `healthdocs-knowledge-source`: Points to the `healthdocs` search index
* `workiq-knowledge-source`: Points to Work IQ for workplace context

In [ ]:
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    SearchIndexFieldReference,
    SearchIndexKnowledgeSource,
    SearchIndexKnowledgeSourceParameters,
    WorkIQKnowledgeSource,
)

HR_KNOWLEDGE_SOURCE_NAME = "hrdocs-knowledge-source"
HEALTH_KNOWLEDGE_SOURCE_NAME = "healthdocs-knowledge-source"
WORK_KNOWLEDGE_SOURCE_NAME = "workiq-knowledge-source"
KNOWLEDGE_SOURCE_NAMES = [
    HR_KNOWLEDGE_SOURCE_NAME,
    HEALTH_KNOWLEDGE_SOURCE_NAME,
    WORK_KNOWLEDGE_SOURCE_NAME,
]

index_client = SearchIndexClient(endpoint=AZURE_SEARCH_SERVICE_ENDPOINT, credential=search_credential)

for knowledge_source_name, index_name, description in [
    (HR_KNOWLEDGE_SOURCE_NAME, HRDOCS_INDEX, "LAB532 HR documents"),
    (HEALTH_KNOWLEDGE_SOURCE_NAME, HEALTHDOCS_INDEX, "LAB532 health benefits documents"),
]:
    knowledge_source = SearchIndexKnowledgeSource(
        name=knowledge_source_name,
        description=description,
        search_index_parameters=SearchIndexKnowledgeSourceParameters(
            search_index_name=index_name,
            source_data_fields=[
                SearchIndexFieldReference(name="uid"),
                SearchIndexFieldReference(name="snippet_parent_id"),
                SearchIndexFieldReference(name="blob_path"),
                SearchIndexFieldReference(name="snippet"),
            ],
            search_fields=[SearchIndexFieldReference(name="snippet")],
            semantic_configuration_name="semantic-configuration",
        ),
    )
    index_client.create_or_update_knowledge_source(knowledge_source=knowledge_source)

work_knowledge_source = WorkIQKnowledgeSource(
    name=WORK_KNOWLEDGE_SOURCE_NAME,
    description="LAB532 Work IQ knowledge source",
)
index_client.create_or_update_knowledge_source(knowledge_source=work_knowledge_source)
print("Knowledge sources created")


## Step 4: Create the multi-source + Work IQ knowledge base

A knowledge base combines the knowledge sources with an LLM and instructions for how retrieval should behave.

For this notebook, the knowledge base uses an `outputMode` of `answerSynthesis` so the attached model can answer the question directly. It also uses a `retrievalReasoningEffort` of `low` so the model can help with query planning and source selection.

In [ ]:
from azure.search.documents.indexes.models import (
    AzureOpenAIVectorizerParameters,
    KnowledgeBase,
    KnowledgeBaseAzureOpenAIModel,
    KnowledgeSourceReference,
)
from azure.search.documents.knowledgebases.models import (
    KnowledgeRetrievalLowReasoningEffort,
    KnowledgeRetrievalOutputMode,
)

KNOWLEDGE_BASE_NAME = "multisource-workiq-knowledge-base"

aoai_params = AzureOpenAIVectorizerParameters(
    resource_url=AZURE_OPENAI_ENDPOINT.rstrip("/") + "/",
    deployment_name=AZURE_OPENAI_CHATGPT_DEPLOYMENT,
    model_name=AZURE_OPENAI_CHATGPT_MODEL_NAME,
    api_key=AZURE_OPENAI_KEY,
)

knowledge_base = KnowledgeBase(
    name=KNOWLEDGE_BASE_NAME,
    description="LAB532 KB combining restored indexes and Work IQ",
    models=[KnowledgeBaseAzureOpenAIModel(azure_open_ai_parameters=aoai_params)],
    knowledge_sources=[KnowledgeSourceReference(name=name) for name in KNOWLEDGE_SOURCE_NAMES],
    retrieval_reasoning_effort=KnowledgeRetrievalLowReasoningEffort(),
    output_mode=KnowledgeRetrievalOutputMode.ANSWER_SYNTHESIS,
    retrieval_instructions="Use Work IQ for workplace context. Use the restored indexes for HR and health policy documents.",
)

index_client.create_or_update_knowledge_base(knowledge_base)
print("Knowledge base created")


## Step 5: Query the knowledge base

Ask a question that can use Work IQ workplace context and the restored search indexes.

The knowledge base uses agentic retrieval to decide when to query the search indexes and when to query the Work IQ source.

In [ ]:
from IPython.display import Markdown, display

from azure.search.documents.knowledgebases import KnowledgeBaseRetrievalClient
from azure.search.documents.knowledgebases.models import (
    KnowledgeBaseMessage,
    KnowledgeBaseMessageTextContent,
    KnowledgeBaseRetrievalRequest,
    SearchIndexKnowledgeSourceParams,
    WorkIQKnowledgeSourceParams,
)

knowledge_base_client = KnowledgeBaseRetrievalClient(
    endpoint=AZURE_SEARCH_SERVICE_ENDPOINT, knowledge_base_name=KNOWLEDGE_BASE_NAME, credential=search_credential
)

question = "Use Work IQ for any relevant workplace context about Gresham Insurance Company Limited and Aviva International Insurance."

retrieval_request = KnowledgeBaseRetrievalRequest(
    messages=[
        KnowledgeBaseMessage(
            role="user",
            content=[KnowledgeBaseMessageTextContent(text=question)],
        )
    ],
    knowledge_source_params=[
        SearchIndexKnowledgeSourceParams(
            knowledge_source_name=HR_KNOWLEDGE_SOURCE_NAME,
            include_references=True,
            include_reference_source_data=True,
            always_query_source=False,
        ),
        SearchIndexKnowledgeSourceParams(
            knowledge_source_name=HEALTH_KNOWLEDGE_SOURCE_NAME,
            include_references=True,
            include_reference_source_data=True,
            always_query_source=False,
        ),
        WorkIQKnowledgeSourceParams(
            knowledge_source_name=WORK_KNOWLEDGE_SOURCE_NAME,
            include_references=True,
            include_reference_source_data=True,
        ),
    ],
    include_activity=True,
    max_runtime_in_seconds=120,
)

result = knowledge_base_client.retrieve(
    retrieval_request=retrieval_request,
    query_source_authorization=QUERY_SOURCE_AUTH,
)

# Extract synthesized answer from response messages
for message in result.response or []:
    for content in message.content or []:
        text = getattr(content, "text", None)
        if text:
            display(Markdown(text))


### Review the references

The next cell prints the raw `references` array returned by the knowledge base so you can inspect which sources contributed to the answer.

In [ ]:
references_json = [ref.as_dict() for ref in result.references or []]
print(json.dumps(references_json, indent=2))


## Bonus: Query from the Copilot CLI

Every knowledge base exposes an MCP server endpoint, which can be added to any MCP-compatible client like VS Code or GitHub Copilot CLI.
If you want to try out querying the knowledge base from the Copilot CLI, use the generated MCP configuration below and follow the steps in the [Copilot CLI sidequest](copilot-cli-sidequest.md).

In [ ]:
mcp_url = f"{AZURE_SEARCH_SERVICE_ENDPOINT}/knowledgebases/{KNOWLEDGE_BASE_NAME}/mcp?api-version={AZURE_SEARCH_API_VERSION}"
headers = {"api-key": AZURE_SEARCH_ADMIN_KEY}
config = {"mcpServers": {KNOWLEDGE_BASE_NAME: {"type": "http", "url": mcp_url, "headers": headers}}}
print(json.dumps(config, indent=2))

➡️ Continue to: [Part 5: Combine Work IQ and Fabric IQ](part5-work-iq-fabric-iq-to-kb.ipynb).